# This notebook was used to support the arguments presented in Chapter 2 of the MSc thesis and, in particular, to study the GRB afterglow emission observed from an off-axis point of view:

Importing the packages used in this notebook:

In [1]:
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib as mpl
import redback as rd
import bilby as bb
import astropy as ast
import pandas as pd
import warnings
import extinction as ex
from astropy.table import Table, Column
from astropy import units as u
from astropy import constants as const
import astropy.io.ascii as ascii
import re
warnings.filterwarnings('ignore')

# Use inline backend
%matplotlib inline

# Packages necessary for the interactive plot to study the GRB afterglow emission light curve under different circumstances:
from ipywidgets import widgets,FloatSlider, HBox, VBox, Play, jslink, interactive_output, Output, ToggleButton, FloatRangeSlider, Layout, HTMLMath
from IPython.display import display, clear_output, Math

# To delete the redback warning and information messages:
import logging
logging.getLogger("redback").setLevel(logging.WARNING)

/tmp/ipykernel_4076643/2341178569.py:2: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.5)
  import scipy as sc
/opt/anaconda3/envs/redback/lib/python3.11/site-packages/lalsimulation/lalsimulation.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal
18:15 bilby INFO    : Running bilby version: 2.7.1
18:15 redback INFO    : Running redback version: 1.15.3


## Then, we directly plot the GRB afterglow emission light curve with interactive widgets of different model parameters to study the afterglow emission observed from a GRB generated in different situations.

In [2]:
# Defining a function to change from wavelengths (in Angstroms) to frequencies (in Hz) that will be useful later on:
def wavetof(wave):
    return(const.c.value/ (wave*1e-10))

# The redshift used is the one of the FXT EP240414a, which is the one studied in this work:
z=0.401
ez=0.005

# Next, assuming a fraction of the shock energy trasmitted to the electrons and magnetic field of:
epse=np.log10(0.1) #electrons
epsb=np.log10(0.01) #magnetic field
# Assumption:
xiN=1 # fraction of electrons that get accelerated. (1=all of them)

#For the half-opening angle of the jet:
thetajet=3

# Models that we will use to plot:

# Tophat jet model:
model='tophat_redback'

# Tophat jet model with tophat cocoon:
#model='twocomponent_redback'
modelcc='twocomponent_redback'

# Tophat jet with powerlaw decay cocoon:
# model='powerlaw_redback'
modelpcc='powerlaw_redback'

# Gaussian structured jet:
#model='gaussian_redback'
modelga='gaussian_redback'


# Times over which we will plot the GRB afterglow emission:
tmin, tmax, N = -8, 8, 10000 
tplot = np.logspace(tmin, tmax, N) #Same as the length of the time array considered


# Now, creating the function to update the parameter space values of the afterglow models
# that we will use connected to the widgets of the figure shown:
def updateparams(loge0,logn0,g0,thvdthc, model, expans=0, wind=0):
    parameters = {}
    parameters['redshift'] = z #Fixed redshift to the EP240414a one (fixed)
    parameters['loge0'] = loge0 # initial isotropic-equivalent kinetic energy of the jet
    parameters['thc'] = np.deg2rad(thetajet) #In degrees for the openning angle of the jet (fixed)
    parameters['thv'] = np.deg2rad(thvdthc*thetajet) # viewing angle as a function of the ratio between the half-opening 
                                                    #angle of the jet and this value
    parameters['logn0']=logn0 # Number density in the ISM-like environment or A star in the wind-like environment
    parameters['xiN']= xiN #Fraction of electrons that get accelerated (fixed)
    parameters['p']=2.5 #Absolute value of the power index of the distribution of lorentz factor in the electrons accelerated (fixed)
    parameters['logepse']=epse #Fraction of shock energy that goes to the electrons (fixed)
    parameters['logepsb']=epsb #Fraction of shock energy that goes to the magnetic field (fixed)
    parameters['g0']=g0 # initial bulk Lorentz factor of the jet
    parameters['expansion']=0 #Initially no lateral expansion of the jet
    #Adding a widget to be able to activate the emission considering the lateral expansion of the jet:
    if expans==True:
        parameters['expansion']=1 
    

    #Wind-like density profile condition:
    if wind==True:
        parameters['k']=2
    
    #Different models conditions:
    if model=='twocomponent_redback':
        parameters['thj'] = np.deg2rad(2.5*np.rad2deg(parameters['thc'])) # Angle that determine the size of the cocoon (fixed)
        parameters['ss']= 0.1 #Fraction of the core energy in the cocoon (fixed)
        parameters['aa']= 0.25*g0 #Lorentz factor outside the core (in the cocoon) (fixed)
    
    if model=='powerlaw_redback':
        parameters['thj'] = np.deg2rad(2.5*np.rad2deg(parameters['thc'])) #Angle that determine the size of the cocoon (fixed)
        parameters['ss']=-3 #power of the decay powerlaw as a function of theta for the kinetic energy in the cocoon (fixed)
        parameters['aa']=-3 #power of the decay powerlaw as a function of theta for gamma in the cocoon (fixed)

    if model=='gaussian_redback':
        parameters['thj'] = np.deg2rad(90) # Theta edge (Angle at which the Gaussian distribution is cut)
        #Comments: Thj is the angle at which the gaussian decay is truncated (The gaussian function extends to infity, so here
        # we truncate it to thj, for angles greater than thj: Jet Ek = 0 and Jet Gamma =1)


    return parameters


# Stablishing the frequencies that will be used as the observing frequencies of the GRB afterglow emission for the simulation:
frequencies = np.array([5*1e9, wavetof(4671.78), wavetof(6141.12), wavetof(7457.89), wavetof(8933.78), 2.418*1e17]) 
# One in radio (~5GHz), one in X-rays (1keV ~ 2.418*10^17 Hz) and for optical data (g r i z)
model_kwargs = {'frequency':frequencies, 'output_format':'flux_density'} #Output format in flux densities
num_data_pts= N # the number of data points to generate

# Defining our function to simulate the GRB afterglow emission
# (the two last arguments only accept 0 for no lateral expansion and constant density of the ISM
# and 1 for lateral expansion and windy ISM):
def simulated_data(tsim, loge0,logn0,g0,thvdthc,modelused, expans=0, wind=0):
    afterglow_simulation = rd.simulate_transients.SimulateGenericTransient(
        model=modelused,
        parameters=updateparams(loge0,logn0,g0,thvdthc, modelused, expans, wind),
        times=tsim,
        data_points=num_data_pts,
        model_kwargs=model_kwargs,
        multiwavelength_transient=True) #Noise term is set by default (required in the GRB afterglow model simulations)
    return afterglow_simulation



#Now, we define different analytic expressions for characteristic times of the GRB afterglow emission light curve to study and understand
#the GRB afterglow emission:

# Defining the used analytic expression for the peak time of the emission at the observer frame and for the on-axis case:
# Expression calculated by Nava et al. 2013 and obtained from Ghirlanda et al. 2018:

def peaktimeon(e0, g0, n0, wind=0):
    s=0
    if wind==1:
        s=2
    #The last term is to convert to the observer frame
    num = (2**(10-2*s)) * np.pi * (4.0 - s) * n0 * const.m_p.cgs.value * (const.c.cgs.value**(5.0 - s))
    den = (17.0 - 4.0*s) * (9.0 - 2.0*s) * (3.0**(2.0 - s)) * e0
    bracket = (g0**(8.0 - 2.0*s)) * (num/den)
    t_peak = bracket ** (-1.0/(3.0 - s)) * (1.0 + z) #Changing with the redshift to the observer frame
    return t_peak/86400 #From seconds to days   

# Defining our analytic expression for the jet break time of the emission, using the equation found on the article 
# Peng et al. 2005.
# In this model, it is considered a tophat jet with a tophat cocoon (the jet break time should be the same for our tophat jet):
# Comment: the formula assumes a uniform ISM (k=0), an on-axis observer (thv/thc=0) and a 'sufficiently' small
# thetajet, otherwise it is not valid.

def jetbreak(e0,n0,thetacore):
    EnergyTerm=(e0/(10**52))**(1/3)
    ISMdensTerm=n0**(-1/3)
    AngleTerm=(np.deg2rad(thetacore)/0.1)**(8/3)
    RedshiftTerm=(1+z) #To convert to observer reference frame
    return 0.25*EnergyTerm*ISMdensTerm*AngleTerm*RedshiftTerm #in days


# Defining our analytic expression for the critical times of the emission considering only synchrotron emission and
# an adiabatic evolution (only adiabatic) as it is considered in the Lamb et al. 2018 paper of the tophat jet model.
# The formulae are obtained from the Sari et al. 1998 paper: "SPECTRA AND LIGHT CURVES OF GAMMA-RAY BURST AFTERGLOWS"

def tmadiabatic(e0,frequency):
    return 0.69*((10**epsb)**(1/3))*((10**epse)**(4/3))*((e0/(1e52))**(1/3))*((frequency/1e15)**(-2/3))* (1.0 + z)

def tcadiabatic(e0, n0, frequency):
    return (7.3*1e-6)*((10**epsb)**(-3))*((e0/(1e52))**(-1))*((frequency/1e15)**(-2))*((n0)**(-2))* (1.0 + z)

#And the same for t0, point at which there exists a transition between fast and slow cooling synchrotron emission regimes (Sari et al. 1998):
def t0adiabatic(e0, n0):
    return 210*((10**epsb)**(2))*((10**epse)**(2))*(e0/(1e52))*n0* (1.0 + z)

#To put latex format names to the widgets:
def labeled(widget, latex):
    out = widgets.Output(layout=Layout(width="150px"))
    with out:
        display(Math(latex))
    return HBox([out, widget])



# Widgets to change the afterglow model parameter values in the interactive plots:
sEk = FloatSlider(value=52, min=48, max=52, step=0.1) 
sn0 = FloatSlider(value=-1, min=-2, max=2, step=0.01) 
sg0 = FloatSlider(value=300, min=2.0, max=1000, step=1) 
sthv = FloatSlider(value=0, min=0, max=10, step=0.01) 
winbut = ToggleButton(value=False,description='Wind-like CSM',disabled=False,button_style='warning', icon='check', tooltip='If on, activates wind-like circumstellar material mode for simulations (k=2), otherwise n_0 will be constant')
sideexp = ToggleButton(value=False,description='Lateral Expansion',disabled=False,button_style='warning', icon='check', tooltip='If on, activates lateral jet expansion for simulations, the jet will expand sideways')
FluxLims= FloatRangeSlider(value=[-10, 2], min=-20, max=5, step=1, readout=True,readout_format='.1f')
TimeLims= FloatRangeSlider(value=[tmin, tmax], min=tmin, max=tmax, step=1, readout=True,readout_format='.1f')





def make_afterglow_from_sim(sim):
    # Creating the 'transient afterglow' object from the simulations objectto plot the simulation results:
    time = sim.data['time'].values
    flux = sim.data['output'].values
    freq = sim.data['frequency'].values
    ferr = sim.data['output_error'].values
    ag = rd.transient.Afterglow(name='aftersim',
                                    time=time, frequency=freq,
                                    flux_density=flux, flux_density_err=ferr,
                                    data_mode='flux_density')
    return ag

# Now, creating directly the plot:
# Use inline backend
%matplotlib inline

# We use an Output widget so the inline plot can be cleared and redrawn each slider change.
out = Output()
controls = VBox([
    HBox([labeled(sEk, r"$\log_{10} (E_{k, 0}/\mathrm{erg})$"), 
          labeled(sn0, r"$\log_{10} (n_0/\mathrm{cm^{-3}})$"), labeled(FluxLims, r"$\log_{10} \mathrm{Flux \, Limits}$")]), 
          HBox([labeled(sg0, r"$\Gamma_0$"), 
                labeled(sthv, r"$\theta_v/\theta_j$"), labeled(TimeLims, r"$\log_{10} \mathrm{Time \, Limits}$")]), 
                HBox([winbut, sideexp])])
display(controls, out)


# Update function that will redraw the figure inside the Output widget
def update_plot(change=None):
    # Reading slider values
    loge0 = sEk.value
    logn0 = sn0.value
    g0 = sg0.value
    thvdthc = sthv.value
    winact=winbut.value
    sideexpact=sideexp.value
    ylimin, ylimax = FluxLims.value
    xlimin, xlimax=TimeLims.value
    # Redrawing inside the Output widget context
    with out:
        clear_output(wait=True)   # To remove the previous image
        fig,ax= plt.subplots(figsize=(9,4))

        # Running the GRB afterglow emission simulation
        sim = simulated_data(tplot, loge0, logn0, g0, thvdthc, model, sideexpact, winact)
        simcc= simulated_data(tplot, loge0, logn0, g0, 0, model,sideexpact, winact) #On-axis only
        #simpcc= simulated_data(tplot, loge0, logn0, g0, thvdthc, modelpcc,sideexpact, winact) #Top-hat with cocoon
        #simga= simulated_data(tplot, loge0, logn0, g0, thvdthc, modelga,sideexpact, winact) #Gaussian structured jet

        ag = make_afterglow_from_sim(sim)
        agcc = make_afterglow_from_sim(simcc) #On-axis only
        #agpcc = make_afterglow_from_sim(simpcc) #Top-hat with cocoon
        #agga = make_afterglow_from_sim(simga) #Gaussian structured jet

        # Plotting using REDBACK codes:
        agcc.plot_data(axes=ax, show=False, band_colors = {
            5e9: 'thistle',
            wavetof(4671.78): 'aliceblue',
            wavetof(6141.12): 'lightcyan',
            wavetof(7457.89): 'azure',
            wavetof(8933.78): 'oldlace',
            2.418e17: 'mistyrose'
        }) #On-axis only
        ag.plot_data(axes=ax, show=False)
        #agpcc.plot_data(axes=ax, show=False) #Top-hat with cocoon
        #agga.plot_data(axes=ax, show=False) #Gaussian structured jet

        # Plotting the peak time:
        #ax.axvline(peaktimeon(10**loge0,g0,10**logn0), color='k',linestyle='--', linewidth=2.0)

        # Plotting the jet break time:
        #ax.axvline(jetbreak(10**loge0,10**logn0, thetajet), color='g',linestyle='--', linewidth=2.0)

        # Plotting tm in different frequencies:
        # ax.axvline(tmadiabatic(10**loge0, frequencies[0]), color='purple',linestyle='--', linewidth=2.0) #Radio
        # ax.axvline(tmadiabatic(10**loge0, frequencies[1]), color='orange',linestyle='--', linewidth=2.0) #g band
        # ax.axvline(tmadiabatic(10**loge0, frequencies[-1]), color='r',linestyle='--', linewidth=2.0) #X-ray

        # Plotting tc in different frequencies:
        #ax.axvline(tcadiabatic(10**loge0, 10**logn0, frequencies[0]), color='magenta',linestyle='--', linewidth=2.0) #Radio
        #ax.axvline(tcadiabatic(10**loge0, 10**logn0, frequencies[1]), color='yellow',linestyle='--', linewidth=2.0) #g band
        #ax.axvline(tcadiabatic(10**loge0, 10**logn0, frequencies[-1]), color='yellow',linestyle='--', linewidth=2.0) #X-ray
        

        # Plotting t0:
        #ax.axvline(t0adiabatic(10**loge0, 10**logn0), color='b',linestyle='--', linewidth=2.0) #X-ray
    
        #Afterglow peak ($t_{dec}$)','Jet Break Time', 'tm in Radio', 'tm in g-band', 'tm in X-rays', 't0', #Additional legends to plot 
        # the different characteristics of the GRB afterglow emission light curve:
        ax.legend(['Radio [5GHz] - Fully On-axis', 'g - Fully On-axis', 'r - Fully On-axis', 'i - Fully On-axis', 'z - Fully On-axis', 'X-rays [1keV] - Fully On-axis','Radio [5GHz]', 'g', 'r', 'i', 'z', 'X-rays [1keV]'],loc='center left', bbox_to_anchor=(1, 0.5)) #'Peak time','Jet Break Time',
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('Observer-Frame Time (d)')
        ax.set_ylabel('Flux density (mJy)')
        ax.set_ylim([10**ylimin, 10**ylimax])
        ax.set_xlim([10**xlimin, 10**xlimax])
        ax.set_title(rf'Simulated GRB afterglow emission using the model: {model} in $REDBACK$')
        fig.tight_layout()
        plt.show()
        plt.close(fig)

# Attaching callbacks to sliders (to update the figure)
for s in (sEk, sn0, sg0, sthv, winbut, sideexp, FluxLims, TimeLims):
    s.observe(update_plot, names='value')


# Initial plot
update_plot()

Output()